<h1><font color="#1a73e8" size="+3">Al-Khwarizmi</font></h1>

Hello,

To start chatting:

- Sign in to your Google account
- Runtime > Change runtime type > T4 GPU (good performance)
- Click the play button below
- Model loads online ~2 mins (nothing is downloaded to device)

In [ ]:
# @title
import warnings
warnings.filterwarnings("ignore")
import logging
logging.disable(logging.WARNING)

import transformers.utils.logging
transformers.utils.logging.set_verbosity_error()
import os, time
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
from io import StringIO
class HiddenPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = StringIO()
    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout = self._original_stdout

from google.colab import output

def clean_text(text):
    text = re.sub(r'<<.*?>>', '', text)
    text = re.sub(r'#### (.*)', r'<div class="final-answer" style="display: block; margin-top: 10px;">\1</div>', text)
    text = re.sub(r'\?\s*\*\*\s*', '? ', text)
    text = re.sub(r'(?<=[\w])(\s*)\*(\s*)(?=[\w])', r'\1\\*\2', text)
    return text

def detect_direction(text):
    """
    Determine base text direction (rtl/ltr) for BiDi rendering of message
    content only. Counts strong-directional characters: Arabic/Hebrew Unicode
    blocks vs Latin letters, ignoring neutral characters (digits, punctuation,
    whitespace). Falls back to 'ltr' when text is empty or has no strong
    directional characters yet (e.g. a message still streaming in).
    """
    if not text:
        return "ltr"
    stripped = re.sub(r'<[^>]+>', '', text)  # ignore HTML tags in the signal
    rtl_chars = re.findall(
        r'[\u0591-\u07FF\uFB1D-\uFDFD\uFE70-\uFEFC]', stripped
    )
    ltr_chars = re.findall(r'[A-Za-z]', stripped)
    if len(rtl_chars) > len(ltr_chars):
        return "rtl"
    return "ltr"

def render_chat(current_user_input=None):
    output.clear(wait=True)
    style = """
    <style>
        .chat-container { line-height: 1.5; }
        .ai-msg, .user-msg {
            font-size: 1.2em !important;
            font-weight: bold !important;
            margin-bottom: 15px;
            display: block;
            text-align: left;
            direction: ltr;
        }
        .ai-msg { color: #1a73e8 !important; }
        .user-msg { color: #35A630 !important; }
        .msg-label {
            direction: ltr;
            unicode-bidi: isolate;
        }
        .msg-content {
            unicode-bidi: plaintext;
        }
        .final-answer {
            color: #1a73e8 !important;
            font-weight: 900 !important;
            font-size: 1.1em;
        }

        .ai-msg span, .user-msg span, .ai-msg p, .user-msg p, .ai-msg div, .user-msg div {
            font-size: 1em !important;
            font-weight: inherit !important;
            display: inline;
            color: inherit !important;
        }
        .final-answer {
            display: block !important;
        }
    </style>
    """
    display(HTML(style))
    display(HTML('<div class="ai-msg"><span class="msg-label">Al-Khwarizmi:</span> <span class="msg-content" dir="ltr">Hi, I am Al-Khwarizmi, how can I help you?</span></div>'))
    for u, a in history:
        u_dir = detect_direction(u)
        display(HTML(f'<div class="user-msg"><span class="msg-label">You:</span> <span class="msg-content" dir="{u_dir}">{u}</span></div>'))
        a_clean = clean_text(a)
        a_dir = detect_direction(a_clean)
        rendered_a = markdown.markdown(a_clean, extensions=["fenced_code", "tables"]).replace("<p>", "").replace("</p>", "")
        display(HTML(f'<div class="ai-msg"><span class="msg-label">Al-Khwarizmi:</span> <span class="msg-content" dir="{a_dir}">{rendered_a}</span></div>'))
    if current_user_input:
        cu_dir = detect_direction(current_user_input)
        display(HTML(f'<div class="user-msg"><span class="msg-label">You:</span> <span class="msg-content" dir="{cu_dir}">{current_user_input}</span></div>'))

def run_chat_loop():
    render_chat()
    while True:
        print("\n")
        user_input = input("You: ")

        if user_input.lower() == "exit":
            break

        render_chat(user_input)

        recent_history = history[-10:]
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        for u, a in recent_history:
            messages.append({"role": "user", "content": u})
            messages.append({"role": "assistant", "content": a})
        messages.append({"role": "user", "content": user_input})

        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        generation_kwargs = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=500,
            temperature=0.7,
            do_sample=True,
        )

        thread = Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()

        reply_id = str(uuid.uuid4())
        display(HTML(f'<div id="{reply_id}" class="ai-msg"><span class="msg-label">Al-Khwarizmi:</span> <span class="msg-content" dir="ltr">...</span></div>'), display_id=reply_id)

        response = ""
        for new_text in streamer:
            response += new_text
            res_clean = clean_text(response)
            res_dir = detect_direction(res_clean)
            rendered = markdown.markdown(res_clean, extensions=["fenced_code", "tables"]).replace("<p>", "").replace("</p>", "")
            update_display(HTML(f'<div class="ai-msg"><span class="msg-label">Al-Khwarizmi:</span> <span class="msg-content" dir="{res_dir}">{rendered}</span></div>'), display_id=reply_id)

        history.append((user_input, response))

try:
    !pip install transformers torch markdown -q

    from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
    from IPython.display import display, HTML, update_display
    import torch
    from threading import Thread
    from datetime import date
    import html, uuid, re
    import markdown

    model_name = "mzoelfakar/Al-Khwarizmi-3B"

    with HiddenPrints():
        transformers.utils.logging.set_verbosity_error()
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16, device_map="auto")

    output.clear()

    SYSTEM_PROMPT = (
    "You are Al-Khwarizmi, named after the  9th-century mathematician whose name is the direct origin of the word 'algorithm'."
    """You are a careful math tutor. Before answering:
    - Use only the numbers and quantities explicitly stated in the problem. Do not introduce, assume, or carry over any value that wasn't given.
    - Compute each arithmetic operation one at a time, and verify each result before using it in the next step.
    - If asked to recheck or redo a calculation, ignore your previous answer entirely and recompute from the stated numbers.
    - Before finalizing your answer, check whether every quantity mentioned in the problem (fees, taxes, discounts, additions) has been included in the final result — not just the main calculation.
    - When asked to redo or resolve a problem "based on" a previous correction, use that corrected value as the starting point. Do not revert to an earlier, uncorrected path."""
    "Reply in the user's last used language."
    "Refrain from repeating unnecessary information."
    "NEVER help with any topics other than math like the weather, cooking, sports, etc."
    f"Today's date is {date.today().strftime('%B %d, %Y')}."
    )

    history = []

    run_chat_loop()
except BaseException:
    pass
finally:
    time.sleep(0.3)
    output.clear()

###### *An AI math tutor named after Muhammad al-Khwarizmi, the 9th-century mathematician whose name is the origin of the word "algorithm".*

###### *Fine-tuned by [Mohamed Zoelfakar](https://www.linkedin.com/in/mzoelfakar/), as part of Hugging Face's [smol-course](https://huggingface.co/learn/smol-course/).*

###### *Note: this model is trained on simple math problems and may make mistakes on complex ones. Model card [available here](https://huggingface.co/mzoelfakar/Al-Khwarizmi-3B).*